# IMPORT LIBRARIES

In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch import optim
from torchvision import models
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import ttach as tta

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

train_df = pd.read_csv("CSVs\\train_images.csv")
val_df = pd.read_csv("CSVs\\val_images.csv")
test_df = pd.read_csv("CSVs\\test_images.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'CSVs\\train_images.csv'

# TRANSFORM IMAGES

## CONFIG

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0

emotion_to_idx = {
    "Anger": 0,
    "Disgust": 1,
    "Fear": 2,
    "Happy": 3,
    "Neutral": 4,
    "Sad": 5
}

## TRANSFORM

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    
    transforms.RandomAffine(
        degrees=0,
        translate=(0.03,0.03),
        scale=(0.95,1.05)
    ),

    transforms.ToTensor(),

    transforms.RandomErasing(
        p=0.35,
        scale=(0.02,0.10),
        ratio=(0.3,3.3)
    ),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])


val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None, device='cpu'):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.device = device

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = row["path"]
        label = emotion_to_idx[row["emotion"]]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

# LOAD DATA

In [ ]:
train_dataset = ImageDataset(train_df, transform=train_transform)
val_dataset = ImageDataset(val_df, transform=val_transform)
test_dataset = ImageDataset(test_df, transform=val_transform)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [ ]:
print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

x, y = next(iter(train_loader))
print("Batch image shape:", x.shape)   # [B,3,224,224]
print("Batch label shape:", y.shape)   # [B]

# CNN MODEL

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

## RESNET

### MODEL

In [ ]:
def resnet34(num_classes=6, pretrained=True):
    model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT if pretrained else None)

    # Freeze all layers first
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze layers from layer2 onwards
    for layer in [model.layer2, model.layer3, model.layer4]:
        for param in layer.parameters():
            param.requires_grad = True

    in_features = model.fc.in_features

    model.fc = nn.Sequential( # type: ignore
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.BatchNorm1d(512),
        nn.Dropout(0.5),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.BatchNorm1d(256),
        nn.Dropout(0.5),
        nn.Linear(256, num_classes),
    )

    return model

model = resnet34().to(device)

### CONFIG

In [ ]:
WEIGHT_DECAY = 1e-3
LEARNING_RATE = 1e-4
PATIENCE = 10
EPOCHS = 100

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters(),
    ),
    weight_decay=WEIGHT_DECAY,
    lr=LEARNING_RATE,
)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS
)

### EVAL FUNC

In [ ]:
def evaluate(loader, eval_model=None):
    if eval_model is None:
        eval_model = model
        
    eval_model.eval()

    y_true = []
    y_pred = []

    total_loss = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = eval_model(images)

            loss = criterion(outputs, labels)
            total_loss += loss.item()

            preds = torch.argmax(
                outputs,
                dim=1
            )

            y_true.extend(
                labels.cpu().numpy()
            )

            y_pred.extend(
                preds.cpu().numpy()
            )

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(y_true, y_pred)

    f1 = f1_score(
        y_true,
        y_pred,
        average="weighted"
    )

    prec = precision_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    rec = recall_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    return avg_loss, acc, f1, prec, rec

### TRAIN

In [ ]:
EPOCHS = 100
PATIENCE = 10

best_val_loss = float("inf")
best_f1 = 0
no_improve = 0

for epoch in range(EPOCHS):

    model.train()
    train_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    val_loss, val_acc, val_f1, val_prec, val_rec = evaluate(
        val_loader
    )

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"LR {current_lr:.6f} | "
        f"Train Loss {train_loss:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Acc {val_acc:.4f} | "
        f"F1 {val_f1:.4f} | "
        f"Prec {val_prec:.4f} | "
        f"Rec {val_rec:.4f}"
    )

    # Save best
    if val_f1 > best_f1:
        best_f1 = val_f1
        no_improve = 0

        torch.save(
            model.state_dict(),
            "best_resnet34_cremad.pth"
        )

    else:
        no_improve += 1

        if no_improve >= PATIENCE:
            print("Early stopping triggered.")
            break

# Load best model and evaluate on test set
model.load_state_dict(torch.load("best_resnet34_cremad.pth"))



In [ ]:
# Evaluate on test set without TTA
test_loss_no_tta, test_acc_no_tta, test_f1_no_tta, test_prec_no_tta, test_rec_no_tta = evaluate(test_loader, model)

print("\nTest Set Performance without TTA:")
print(f"Test Loss: {test_loss_no_tta:.4f}")
print(f"Test Accuracy: {test_acc_no_tta:.4f}")
print(f"Test F1 Score: {test_f1_no_tta:.4f}")
print(f"Test Precision: {test_prec_no_tta:.4f}")
print(f"Test Recall: {test_rec_no_tta:.4f}")

### TEST

In [ ]:
# TTA
tta_model = tta.ClassificationTTAWrapper(model, tta.aliases.five_crop_transform(224, 224))
test_loss, test_acc, test_f1, test_prec, test_rec = evaluate(test_loader, tta_model)

print("\nTest Set Performance with TTA:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")
print(f"Test Precision: {test_prec:.4f}")
print(f"Test Recall: {test_rec:.4f}")